# Data Preparation for TB Resistance Forecasting

This notebook builds the feature-complete variant tables used in the manuscript analyses. It processes WHO 2021 and WHO 2023 catalogue entries, standardizes amino-acid substitutions, generates mutated protein sequences, and assembles the multimodal features used downstream for forecasting.

## What this notebook does
- loads and standardizes the WHO 2021 and WHO 2023 catalogues
- converts mutations to one-letter amino-acid notation
- generates mutated protein FASTA sequences for each variant
- computes structural proximity, Rosetta, language-model, and AAIndex-derived features
- writes year-specific derived feature tables for model training and forecasting

## Main inputs
- `data/catalog/WHO-UCN-GTB-PCI-2021.7-eng.xlsx`
- `data/catalog/WHO-UCN-TB-2023.7-eng.xlsx`
- `data/catalog/protein_sequences.csv`
- `data/catalog/AAIndex_PCA.csv`
- `data/distmaps/`
- `data/Rosetta/refined/`

## Main outputs
- `data/derived_features/2021/2021_final_df.csv`
- `data/derived_features/2023/2023_final_df.csv`
- intermediate standardized catalogues and mutated sequence files used to build those final tables

## How this notebook fits into the workflow
Run this notebook before `model_essential_nonessential_combined.ipynb`. The model notebook expects the final derived feature tables produced here.


## Environment and path setup

The next cell detects the code directory and the project data directory automatically, so the notebook can be run either from the release folder itself or from a larger repository checkout. It also adds the local code directory to `sys.path` so that `forecast_utils.py` is imported from this release package.


In [ ]:
import pandas as pd
import numpy as np
import re
import csv
import os
import sys
import torch
import esm
from pathlib import Path
from Bio import SeqIO
from evcouplings.compare import DistanceMap


# Locate the release-code directory that contains the shared utility module.
def find_run_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "forecast_utils.py").exists() and (parent / "requirements.txt").exists():
            return parent
    raise FileNotFoundError("Could not locate code root containing forecast_utils.py and requirements.txt")


# Locate the repository-level data directory used by the manuscript workflow.
def find_project_root(run_root: Path) -> Path:
    if (run_root / "data").exists():
        return run_root
    if (run_root.parent / "data").exists():
        return run_root.parent
    raise FileNotFoundError("Could not locate project root containing data/")


RUN_ROOT = find_run_root(Path.cwd().resolve())
PROJECT_ROOT = find_project_root(RUN_ROOT)
os.chdir(RUN_ROOT)
if str(RUN_ROOT) not in sys.path:
    sys.path.insert(0, str(RUN_ROOT))


from forecast_utils import *


## Build the 2021 feature table

This section prepares the 2021 WHO catalogue for supervised training. It standardizes the catalogue, writes a preprocessed mutation table, generates mutated FASTA sequences, and then assembles the multimodal feature table used for downstream model fitting.


In [ ]:
from forecast_utils import PrepTools, MutationTools, FeaturePipeline, CatalogNormalizer

# 1) Load protein sequences
protein_sequences_df = PrepTools.load_protein_sequences(PROJECT_ROOT / "data/catalog/protein_sequences.csv")
genes_of_interest = protein_sequences_df['gene'].unique()

# 2) Load + standardize WHO catalog
standardized_catalog = CatalogNormalizer.load_and_standardize(
    PROJECT_ROOT / "data/catalog/WHO-UCN-GTB-PCI-2021.7-eng.xlsx",
    year=2021,
    genes_of_interest=genes_of_interest
)

# 3) Save preprocessed
PrepTools.save_preprocessed(
    standardized_catalog,
    RUN_ROOT / "data/derived_features/2021/2021_mutations_with_one_letter_all_confidence.csv"
)

# 4) Generate mutated FASTAs
results, mismatches = MutationTools.generate_mutated_fastas(
    protein_sequences_df,
    standardized_catalog,
    output_dir=RUN_ROOT / "mutated_sequences_2021"
)
print(f"{len(results)} mutations applied OK")
print(f"{len(mismatches)} problems logged")


In [ ]:
# 5) Run full feature pipeline
final_df = FeaturePipeline.run(
    catalog_df=standardized_catalog,
    protein_df=protein_sequences_df,
    fasta_dir=RUN_ROOT / "mutated_sequences_2021",
    distmap_dir=PROJECT_ROOT / "data/distmaps/",
    protein_details_path=PROJECT_ROOT / "data/catalog/17_proteins_details.xlsx",
    rosetta_dir=PROJECT_ROOT / "data/Rosetta/refined",
    aaindex_path=PROJECT_ROOT / "data/catalog/AAIndex_PCA.csv",
    esm_model_path="/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt",
    out_dir=RUN_ROOT / "data/derived_features/2021",
    top_k_llr=10
)

print(final_df.shape)
print(final_df.head())


In [ ]:
rename_map = {
    # Mutation info
    "one_letter_mutation": "mutation_oneletter",
    "Wildtype_AA": "mutation_wt",
    "position": "mutation_pos",
    "Mutated_AA": "mutation_mut",
    # Frequency
    "frequency": "freq_variant",
    # Rosetta
    "fa_atr": "Rosetta_fa_atr",
    "fa_rep": "Rosetta_fa_rep",
    "fa_sol": "Rosetta_fa_sol",
    "fa_elec": "Rosetta_fa_elec",
    "fa_dun": "Rosetta_fa_dun",
    "thermostability": "Rosetta_ddG",
    # Delta-Z
    "delta_z": "DeltaZ",
    # Proximity
    "WHO_Adjusted_Position": "Prox_WHO_Adjusted_Pos",
    "Proximity_1D": "Prox_1D",
    "Nearest_1D_Index": "Prox_1D_nearest",
    "Proximity_to_R_Conferring": "Prox_3D",
    "Nearest_Mutation_Index": "Prox_3D_nearest",
    "Proximity_to_R_Conferring_zeroed": "Prox_3D_zeroed",
    # LLR
    "llr_score": "LLR_score",
}

# Apply rename
final_df = final_df.rename(columns=rename_map)

# Expand renaming for AAIndex
for i in range(1, 9):
    final_df = final_df.rename(columns={
        f"mut_AAIndex{i}": f"AAIndex_mut{i}",
        f"delta_AAIndex{i}": f"AAIndex_delta{i}"
    })

# Expand renaming for LLR expanded dims
for c in final_df.columns:
    if c.startswith("expanded_llr_dim_"):
        dim = c.split("_")[-1]
        final_df = final_df.rename(columns={c: f"LLR_dim{dim}"})

# Define ordered groups
ordered_cols = [
    "gene", "drug", "confidence", "phenotype",
    "mutation_oneletter", "mutation_wt", "mutation_pos", "mutation_mut",
    "freq_variant",
    "Rosetta_fa_atr", "Rosetta_fa_rep", "Rosetta_fa_sol", "Rosetta_fa_elec",
    "Rosetta_fa_dun", "Rosetta_ddG",
    "DeltaZ",
    "Prox_WHO_Adjusted_Pos", "Prox_1D", "Prox_1D_nearest",
    "Prox_3D", "Prox_3D_nearest", "Prox_3D_zeroed",
    "LLR_score"
] + sorted([c for c in final_df.columns if c.startswith("LLR_dim")],
           key=lambda x: int(x.replace("LLR_dim",""))) \
  + sorted([c for c in final_df.columns if c.startswith("AAIndex_")])

# Reorder (and keep extras at end if any)
final_df = final_df[[c for c in ordered_cols if c in final_df.columns] +
                    [c for c in final_df.columns if c not in ordered_cols]]


In [ ]:
final_df.to_csv(RUN_ROOT / 'data/derived_features/2021/2021_final_df.csv', index=False)

### 2023

In [ ]:
from forecast_utils import CatalogNormalizer, MutationTools, PrepTools

# 1) Load proteins
protein_sequences_df = PrepTools.load_protein_sequences(PROJECT_ROOT / "data/catalog/protein_sequences.csv")
genes_of_interest = protein_sequences_df['gene'].unique()

# 2) Standardize catalog (2023)
cat2023 = CatalogNormalizer.load_and_standardize(
    PROJECT_ROOT / "data/catalog/WHO-UCN-TB-2023.7-eng.xlsx",
    year=2023,
    genes_of_interest=genes_of_interest
)

# 3) Save standardized output
cat2023.to_csv(RUN_ROOT / "data/derived_features/2023/2023_mutations_with_one_letter_all_confidence.csv", index=False)

# 4) Generate mutated FASTAs
results, mismatches = MutationTools.generate_mutated_fastas(
    protein_sequences_df,
    cat2023,
    output_dir=RUN_ROOT / "mutated_sequences_2023"
)

print(f"{len(results)} mutations applied OK")
print(f"{len(mismatches)} problems logged")


In [ ]:
# 5) Run full feature pipeline
final_df = FeaturePipeline.run(
    catalog_df=cat2023,
    protein_df=protein_sequences_df,
    fasta_dir=RUN_ROOT / "mutated_sequences_2023",
    distmap_dir=PROJECT_ROOT / "data/distmaps/",
    protein_details_path=PROJECT_ROOT / "data/catalog/17_proteins_details.xlsx",
    rosetta_dir=PROJECT_ROOT / "data/Rosetta/refined",
    aaindex_path=PROJECT_ROOT / "data/catalog/AAIndex_PCA.csv",
    esm_model_path="/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt",
    out_dir=RUN_ROOT / "data/derived_features/2023",
    top_k_llr=10
)

print(final_df.shape)
print(final_df.head())


In [ ]:
rename_map = {
    # Mutation info
    "one_letter_mutation": "mutation_oneletter",
    "Wildtype_AA": "mutation_wt",
    "position": "mutation_pos",
    "Mutated_AA": "mutation_mut",
    # Frequency
    "frequency": "freq_variant",
    # Rosetta
    "fa_atr": "Rosetta_fa_atr",
    "fa_rep": "Rosetta_fa_rep",
    "fa_sol": "Rosetta_fa_sol",
    "fa_elec": "Rosetta_fa_elec",
    "fa_dun": "Rosetta_fa_dun",
    "thermostability": "Rosetta_ddG",
    # Delta-Z
    "delta_z": "DeltaZ",
    # Proximity
    "WHO_Adjusted_Position": "Prox_WHO_Adjusted_Pos",
    "Proximity_1D": "Prox_1D",
    "Nearest_1D_Index": "Prox_1D_nearest",
    "Proximity_to_R_Conferring": "Prox_3D",
    "Nearest_Mutation_Index": "Prox_3D_nearest",
    "Proximity_to_R_Conferring_zeroed": "Prox_3D_zeroed",
    # LLR
    "llr_score": "LLR_score",
}

# Apply rename
final_df = final_df.rename(columns=rename_map)

# Expand renaming for AAIndex
for i in range(1, 9):
    final_df = final_df.rename(columns={
        f"mut_AAIndex{i}": f"AAIndex_mut{i}",
        f"delta_AAIndex{i}": f"AAIndex_delta{i}"
    })

# Expand renaming for LLR expanded dims
for c in final_df.columns:
    if c.startswith("expanded_llr_dim_"):
        dim = c.split("_")[-1]
        final_df = final_df.rename(columns={c: f"LLR_dim{dim}"})

# Define ordered groups
ordered_cols = [
    "gene", "drug", "confidence", "phenotype",
    "mutation_oneletter", "mutation_wt", "mutation_pos", "mutation_mut",
    "freq_variant",
    "Rosetta_fa_atr", "Rosetta_fa_rep", "Rosetta_fa_sol", "Rosetta_fa_elec",
    "Rosetta_fa_dun", "Rosetta_ddG",
    "DeltaZ",
    "Prox_WHO_Adjusted_Pos", "Prox_1D", "Prox_1D_nearest",
    "Prox_3D", "Prox_3D_nearest", "Prox_3D_zeroed",
    "LLR_score"
] + sorted([c for c in final_df.columns if c.startswith("LLR_dim")],
           key=lambda x: int(x.replace("LLR_dim",""))) \
  + sorted([c for c in final_df.columns if c.startswith("AAIndex_")])

# Reorder (and keep extras at end if any)
final_df = final_df[[c for c in ordered_cols if c in final_df.columns] +
                    [c for c in final_df.columns if c not in ordered_cols]]


In [ ]:
final_df

In [ ]:
final_df.to_csv(RUN_ROOT / 'data/derived_features/2023/2023_final_df.csv', index=False)